# 02 Chunker -- Semantic Parent-Child Chunking

## Purpose

This notebook demonstrates the **text chunking** step of the ImmunoBiology RAG pipeline. Raw page-level documents from `01_pdf_parser` are split into a two-level parent-child hierarchy optimized for retrieval.

## Position in Pipeline

```
01_pdf_parser --> [02_chunker] --> 03_embedder --> 04_retriever
```

## Architecture: Parent-Child Chunking

The system uses a two-level design (adapted from the Tesla RAG architecture):

1. **Parent chunks** (~512 tokens): Produced by the semantic chunking service. These provide complete context and are what the LLM reads.
2. **Child chunks** (<= 512 tokens with 100-token overlap): Sub-splits of parents using `RecursiveCharacterTextSplitter`. These are used for precise vector retrieval.

At retrieval time, child chunks are matched to the query, but the corresponding parent chunks are passed to the LLM for richer context.

## Learning Objectives

1. Understand the semantic chunking service (FastAPI on port 6000) and its role in grouping paragraphs
2. See how `RecursiveCharacterTextSplitter` creates overlapping child chunks
3. Compare parent vs. child chunks side by side
4. Analyze chunk length distribution to verify parameter calibration
5. Understand how chunks are stored in MongoDB with parent-child relationships

## Imports

We need `tiktoken` for accurate token counting (cl100k_base, same tokenizer family as GPT-4/LLaMA), `RecursiveCharacterTextSplitter` from LangChain, and the project's chunker module.

In [ ]:
import sys
import json
import copy
import hashlib
import pickle
from pathlib import Path
from typing import List

import tiktoken
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import constant
from src.client.semantic_chunk_client import request_semantic_chunk

# Tokenizer for accurate token counts
encoding = tiktoken.get_encoding("cl100k_base")

print(f"Project root: {PROJECT_ROOT}")
print(f"Chunk size: {constant.chunk_size} tokens")
print(f"Chunk overlap: {constant.chunk_overlap} tokens")
print(f"Separators: {constant.chunk_separators}")
print(f"Semantic service URL: {constant.semantic_service_url}")

## Load Parsed JSON Pages

We load the page-level Documents produced by `01_pdf_parser`. These are cached as a pickle file by `build_index.py`, or we can reconstruct them from the per-page JSON files in `data/processed/`.

In [ ]:
import os

raw_docs = []

# Method 1: Load from pickle cache (fastest)
if os.path.exists(constant.raw_docs_path):
    with open(constant.raw_docs_path, "rb") as f:
        raw_docs = pickle.load(f)
    print(f"Loaded {len(raw_docs)} page documents from pickle cache.")

# Method 2: Reconstruct from JSON files
else:
    processed_dir = Path(constant.processed_dir)
    json_files = sorted(processed_dir.rglob("text/page_*.json"))
    if json_files:
        for jf in json_files:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
            raw_docs.append(Document(
                page_content=data["page_content"],
                metadata=data["metadata"]
            ))
        print(f"Loaded {len(raw_docs)} page documents from JSON files.")
    else:
        print("No parsed documents found. Run 01_pdf_parser.ipynb first.")

if raw_docs:
    sample = raw_docs[0]
    print(f"\nSample page metadata:")
    print(f"  source_file: {sample.metadata.get('source_file')}")
    print(f"  chapter:     {sample.metadata.get('chapter')}")
    print(f"  page:        {sample.metadata.get('page')}")
    print(f"  text length: {len(sample.page_content)} chars, "
          f"{len(encoding.encode(sample.page_content))} tokens")

## Semantic Chunking Service

The semantic chunking service runs as a FastAPI server on **port 6000** (defined in `config.yaml`). It uses the `all-MiniLM-L6-v2` embedding model to group consecutive paragraphs that are semantically similar.

**How it works:**
1. Input text is split into paragraphs
2. Each paragraph is embedded with MiniLM
3. Consecutive paragraphs with high cosine similarity are grouped together
4. Groups are capped at `semantic_group_size` (15) paragraphs

If the service is unavailable, the client falls back to returning the text unsplit.

**Start the service before running this cell:**
```bash
python src/server/semantic_chunk.py
```

In [ ]:
# Demonstrate semantic chunking on a single page
if raw_docs:
    sample_text = raw_docs[0].page_content
    print(f"Input text length: {len(sample_text)} chars")
    print(f"Input tokens: {len(encoding.encode(sample_text))}")
    
    # Call the semantic chunking service
    semantic_groups = request_semantic_chunk(sample_text, group_size=constant.semantic_group_size)
    
    print(f"\nSemantic chunking produced {len(semantic_groups)} group(s):")
    for i, group in enumerate(semantic_groups):
        tokens = len(encoding.encode(group))
        print(f"  Group {i+1}: {len(group)} chars, {tokens} tokens")
        print(f"    Preview: {group[:120]}...")
else:
    print("No documents loaded.")

## Child Chunk Splitting with RecursiveCharacterTextSplitter

Each semantic group (parent) is further split into child chunks using LangChain's `RecursiveCharacterTextSplitter`. The splitter:
1. Tries separators in priority order: `\n\n` > `\n` > `. ` > ` ` > `""`
2. Ensures each chunk is at most `chunk_size` tokens (512)
3. Maintains `chunk_overlap` tokens (100) between consecutive chunks

Token counting uses `cl100k_base` (same tokenizer as GPT-4) for accurate measurement.

In [ ]:
# Create the text splitter with project configuration
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=constant.chunk_size,
    chunk_overlap=constant.chunk_overlap,
    separators=constant.chunk_separators,
    length_function=lambda text: len(encoding.encode(text)),
)

# Split the first semantic group into child chunks
if raw_docs:
    semantic_groups = request_semantic_chunk(raw_docs[0].page_content)
    if semantic_groups:
        parent_text = semantic_groups[0]
        child_chunks = text_splitter.split_text(parent_text)
        
        print(f"Parent group: {len(encoding.encode(parent_text))} tokens")
        print(f"Child chunks produced: {len(child_chunks)}")
        
        for i, chunk in enumerate(child_chunks):
            tokens = len(encoding.encode(chunk))
            print(f"\n  Child {i+1}: {tokens} tokens, {len(chunk)} chars")
            print(f"    {chunk[:150]}...")

## Parent vs. Child Chunk Comparison

This cell shows parent and child chunks side by side to illustrate the hierarchy. The parent provides complete context; children are precise retrieval units.

In [ ]:
if raw_docs:
    # Process a few pages to get parent-child examples
    semantic_groups = request_semantic_chunk(raw_docs[0].page_content)
    
    if semantic_groups and len(semantic_groups) > 0:
        parent_text = semantic_groups[0]
        child_chunks = text_splitter.split_text(parent_text)
        
        print("=" * 70)
        print("PARENT CHUNK (semantic group -- what the LLM sees)")
        print("=" * 70)
        print(f"Tokens: {len(encoding.encode(parent_text))}")
        print(f"\n{parent_text[:600]}")
        if len(parent_text) > 600:
            print("...")
        
        print(f"\n{'=' * 70}")
        print(f"CHILD CHUNKS (retrieval units -- what the vector search matches)")
        print(f"{'=' * 70}")
        
        for i, chunk in enumerate(child_chunks[:3]):
            tokens = len(encoding.encode(chunk))
            print(f"\n--- Child {i+1} ({tokens} tokens) ---")
            print(chunk[:300])
            if len(chunk) > 300:
                print("...")
        
        if len(child_chunks) > 3:
            print(f"\n... and {len(child_chunks) - 3} more child chunk(s)")
else:
    print("No documents loaded.")

## Chunk Length Distribution

Visualizing the chunk length distribution verifies that `chunk_size` and `chunk_overlap` are well-calibrated. We expect:
- Most chunks near 512 tokens (the target)
- Some shorter chunks from paragraph boundaries
- No chunks significantly exceeding 512 tokens

In [ ]:
from src.chunker import texts_split

# Run chunking on all documents (dry_run=True skips MongoDB writes)
if raw_docs:
    print(f"Chunking {len(raw_docs)} pages (dry_run mode -- no MongoDB writes)...")
    split_docs = texts_split(raw_docs, dry_run=True)
    
    # Compute token lengths for all chunks
    chunk_lengths = [len(encoding.encode(doc.page_content)) for doc in split_docs]
    
    print(f"\nTotal chunks produced: {len(split_docs)}")
    print(f"Token stats: min={min(chunk_lengths)}, "
          f"median={sorted(chunk_lengths)[len(chunk_lengths)//2]}, "
          f"max={max(chunk_lengths)}, "
          f"mean={sum(chunk_lengths)/len(chunk_lengths):.0f}")
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(chunk_lengths, bins=50, edgecolor="black", color="steelblue", alpha=0.8)
    axes[0].axvline(constant.chunk_size, color="red", linestyle="--", linewidth=1.5,
                    label=f"chunk_size={constant.chunk_size}")
    axes[0].set_title("Chunk Length Distribution (tokens)", fontsize=13)
    axes[0].set_xlabel("Tokens per chunk")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    
    stats_text = (
        f"n={len(chunk_lengths)}\n"
        f"min={min(chunk_lengths)}\n"
        f"median={sorted(chunk_lengths)[len(chunk_lengths)//2]}\n"
        f"max={max(chunk_lengths)}\n"
        f"mean={sum(chunk_lengths)/len(chunk_lengths):.0f}"
    )
    axes[0].text(0.98, 0.97, stats_text, transform=axes[0].transAxes,
                 verticalalignment="top", horizontalalignment="right",
                 fontsize=9, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    
    # Cumulative distribution
    sorted_lens = sorted(chunk_lengths)
    cumulative = [i / len(sorted_lens) for i in range(len(sorted_lens))]
    axes[1].plot(sorted_lens, cumulative, color="steelblue")
    axes[1].axvline(constant.chunk_size, color="red", linestyle="--", linewidth=1.5,
                    label=f"chunk_size={constant.chunk_size}")
    axes[1].set_title("Cumulative Distribution", fontsize=13)
    axes[1].set_xlabel("Tokens per chunk")
    axes[1].set_ylabel("Cumulative fraction")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No documents to chunk.")

## Save to MongoDB

In production, chunks are stored in MongoDB with parent-child relationships. Each record contains:
- `unique_id`: MD5 hash of the chunk text (primary key)
- `page_content`: the chunk text
- `metadata`: source_file, chapter, page, parent_id, chunk_id, etc.

Child chunks have a `parent_id` field linking back to their parent. This link is used by `merge_docs()` at retrieval time to fetch the full parent context.

**Note**: Requires a running MongoDB instance at `localhost:27017` (configured in `config.yaml`).

In [ ]:
# Full chunking with MongoDB storage (set dry_run=False)
# Uncomment the lines below to actually write to MongoDB.

# from src.chunker import texts_split
# if raw_docs:
#     split_docs = texts_split(raw_docs, dry_run=False)
#     print(f"Stored {len(split_docs)} chunks in MongoDB.")
#     
#     # Cache for next step (embedder)
#     with open(constant.split_docs_path, "wb") as f:
#         pickle.dump(split_docs, f)
#     print(f"Cached to {constant.split_docs_path}")

print("MongoDB write is commented out by default.")
print("Uncomment the code above to persist chunks to MongoDB.")
print(f"\nMongoDB config:")
print(f"  Host:       {constant.mongo_host}:{constant.mongo_port}")
print(f"  Database:   {constant.mongo_db_name}")
print(f"  Collection: {constant.mongo_collection}")

## Summary

### Outputs Produced

| Output | Location | Description |
|--------|----------|-------------|
| Parent chunks | MongoDB `chunk_metadata` | Semantic groups (~512 tokens), used as LLM context |
| Child chunks | MongoDB `chunk_metadata` | Sub-splits with overlap, used for vector retrieval |
| Chunk histogram | `outputs/diagnostics/chunk_length_dist.png` | Token length distribution |
| Split docs pickle | `data/processed_docs/split_docs.pkl` | Cached for embedder |

### Key Parameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `chunk_size` | 512 tokens | English academic text; ~15-20 sentences per chunk |
| `chunk_overlap` | 100 tokens | 20% overlap prevents cutting mid-concept |
| `semantic_group_size` | 15 | English paragraphs are longer than Chinese |
| `max_parent_size` | 512 chars | Parents smaller than this are also retrieval units |

### Next Notebook

Proceed to **`03_embedder.ipynb`** to encode chunks into BGE-M3 dense vectors and build BM25 + Chroma indices.